# AI4Lassa — 02. Model Benchmarking (Phase 3)

Benchmarks the existing app's model architecture (linear SVM) against a range of
regression and classification models for 1-month-ahead case-count forecasting.

Evaluation on the **validation set (2022–2023) only** — the test set (2024–2025) stays
untouched until the final phase.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.svm import SVR, SVC
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

DATA_PATH = "../data/processed/monthly_features.csv"
RISK_THRESHOLD = 461.6  # from feature-engineering notebook: train mean + 1.5*SD

FEATURE_COLS = [
    "case_count",
    "case_count_lag1", "case_count_lag2", "case_count_lag3",
    "case_count_lag6", "case_count_lag12",
    "case_count_roll3_mean", "case_count_roll6_mean", "case_count_roll3_max",
    "case_growth_lag1", "positivity_rate_lag1",
    "month_sin", "month_cos", "year",
]
TARGET_COL = "target_next_month_cases"

## Load the temporal train / validation / test split

In [2]:
df = pd.read_csv(DATA_PATH)
df["month_ts"] = pd.to_datetime(df["month_ts"])

train = df[(df["month_ts"] >= "2016-01-01") & (df["month_ts"] <= "2021-12-31")].reset_index(drop=True)
val   = df[(df["month_ts"] >= "2022-01-01") & (df["month_ts"] <= "2023-12-31")].reset_index(drop=True)
test  = df[(df["month_ts"] >= "2024-01-01")].reset_index(drop=True)

for split in (train, val, test):
    split["high_risk"] = (split[TARGET_COL] > RISK_THRESHOLD).astype(int)

print(f"Train {len(train)} | Val {len(val)} | Test {len(test)} (test untouched this phase)")

Train 72 | Val 24 | Test 23 (test untouched this phase)


## Regression benchmark: naive baseline, architecture-matched SVM, and 5 alternatives

In [3]:
Xtr, ytr = train[FEATURE_COLS], train[TARGET_COL]
Xval, yval = val[FEATURE_COLS], val[TARGET_COL]

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xval_s = scaler.transform(Xtr), scaler.transform(Xval)

results = []
results.append(("Naive persistence (t+1 = t)", yval, val["case_count"].values))

svr = SVR(kernel="linear").fit(Xtr_s, ytr)
results.append(("SVM (linear, architecture-matched baseline)", yval, svr.predict(Xval_s)))

lr = LinearRegression().fit(Xtr_s, ytr)
results.append(("Linear Regression", yval, lr.predict(Xval_s)))

rf = RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=3, random_state=42).fit(Xtr, ytr)
results.append(("Random Forest", yval, rf.predict(Xval)))

xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, random_state=42).fit(Xtr, ytr)
results.append(("XGBoost", yval, xgb_model.predict(Xval)))

lgb_model = lgb.LGBMRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                               min_child_samples=5, verbosity=-1, random_state=42).fit(Xtr, ytr)
results.append(("LightGBM", yval, lgb_model.predict(Xval)))

cb_model = cb.CatBoostRegressor(iterations=200, depth=3, learning_rate=0.05, verbose=False, random_state=42).fit(Xtr, ytr)
results.append(("CatBoost", yval, cb_model.predict(Xval)))

rows = []
for name, y_true, y_pred in results:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    rows.append({"Model": name, "MAE": mae, "RMSE": rmse, "MAPE_%": mape, "R2": r2})

reg_results = pd.DataFrame(rows).sort_values("MAE")
reg_results.to_csv("../outputs/metrics/regression_benchmark_val.csv", index=False)
reg_results

,Model,MAE,RMSE,MAPE_%,R2
3,Random Forest,55.592179,81.163694,16.052408,0.609467
6,CatBoost,60.590352,82.721083,17.535714,0.594336
5,LightGBM,70.014672,88.944794,21.819973,0.530998
4,XGBoost,70.199034,92.113602,20.758313,0.496985
2,Linear Regression,71.468507,91.419871,25.949802,0.504533
0,Naive persistence (t+1 = t),80.000000,111.345184,24.375236,0.265018
1,"SVM (linear, architecture-matched baseline)",83.372716,122.018323,22.613482,0.117360


**Finding:** every model except the linear-SVM baseline beats the naive persistence
baseline, confirming the engineered features add real signal. Random Forest leads.

## Classification benchmark: derived high-risk flag

In [4]:
Xtr, ytr = train[FEATURE_COLS], train["high_risk"]
Xval, yval = val[FEATURE_COLS], val["high_risk"]

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xval_s = scaler.transform(Xtr), scaler.transform(Xval)

print(f"Train high-risk months: {ytr.sum()}/{len(ytr)}  |  Val high-risk months: {yval.sum()}/{len(yval)}")

models = {
    "SVM (linear, architecture-matched baseline)": SVC(kernel="linear", probability=True, class_weight="balanced", random_state=42),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                  scale_pos_weight=(len(ytr) - ytr.sum()) / max(ytr.sum(), 1),
                                  random_state=42, eval_metric="logloss"),
    "LightGBM": lgb.LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, class_weight="balanced", verbosity=-1, random_state=42),
    "CatBoost": cb.CatBoostClassifier(iterations=200, depth=3, learning_rate=0.05, auto_class_weights="Balanced", verbose=False, random_state=42),
}

rows = []
for name, model in models.items():
    if name.startswith("SVM") or name == "Logistic Regression":
        model.fit(Xtr_s, ytr); pred = model.predict(Xval_s); proba = model.predict_proba(Xval_s)[:, 1]
    else:
        model.fit(Xtr, ytr); pred = model.predict(Xval); proba = model.predict_proba(Xval)[:, 1]

    row = {
        "Model": name,
        "Accuracy": accuracy_score(yval, pred),
        "Recall": recall_score(yval, pred, zero_division=0),
        "Precision": precision_score(yval, pred, zero_division=0),
        "F1": f1_score(yval, pred, zero_division=0),
        "Brier": brier_score_loss(yval, proba),
    }
    if yval.nunique() > 1:
        row["ROC_AUC"] = roc_auc_score(yval, proba)
        row["PR_AUC"] = average_precision_score(yval, proba)
    else:
        row["ROC_AUC"] = np.nan; row["PR_AUC"] = np.nan
    rows.append(row)

clf_results = pd.DataFrame(rows).sort_values("Recall", ascending=False)
clf_results.to_csv("../outputs/metrics/classification_benchmark_val.csv", index=False)
clf_results

Train high-risk months: 5/72  |  Val high-risk months: 3/24


,Model,Accuracy,Recall,Precision,F1,Brier,ROC_AUC,PR_AUC
0,"SVM (linear, architecture-matched baseline)",0.708333,1.000000,0.300000,0.461538,0.086408,0.968254,0.866667
1,Logistic Regression,0.750000,1.000000,0.333333,0.500000,0.155374,0.968254,0.805556
3,XGBoost,0.833333,0.333333,0.333333,0.333333,0.091893,0.920635,0.666667
4,LightGBM,0.833333,0.333333,0.333333,0.333333,0.094442,0.904762,0.500000
2,Random Forest,0.875000,0.000000,0.000000,0.000000,0.071433,0.952381,0.833333
5,CatBoost,0.833333,0.000000,0.000000,0.000000,0.089488,0.920635,0.555556


**Finding — read carefully:** Random Forest has the *best accuracy* on this task but
**recall = 0** — it simply predicts "not high-risk" every month, including the 3 real
high-risk months in validation. This is a textbook illustration of why accuracy alone is
the wrong metric for a rare-event early-warning task. The two linear models (SVM, Logistic
Regression) caught every real high-risk month, at the cost of more false alarms.